### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [38]:
# No environment variables needed for local Ollama setup


In [39]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

import chromadb
from urllib.request import urlopen
import json


In [40]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### 1. Sample Data

In [41]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [42]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document create in : {temp_dir}")

Sample document create in : /var/folders/8m/3mlf420j3_54kpky2m3w1cm00000gn/T/tmpc0ncqp3l


In [43]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)



In [44]:
temp_dir

'/var/folders/8m/3mlf420j3_54kpky2m3w1cm00000gn/T/tmp0gzi9pw_'

### 2. Document Loading

In [45]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents from directory
loader = DirectoryLoader(
    "data", 
    glob="*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")


Loaded 3 documents

First document preview:

    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity r...


In [46]:
documents

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='\n    Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.\n    '),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models,

### Document Splitting

In [47]:
# Initialize text splitter
# Better separators preserve semantic boundaries and improve retrieval quality.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=80,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")


Created 9 chunks from 3 documents

Chunk example:
Content: Natural Language Processing (NLP)...
Metadata: {'source': 'data/doc_2.txt'}


In [48]:
chunks

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n   

### Embedding Models

In [49]:
# Local Ollama configuration
OLLAMA_BASE_URL = "http://localhost:11434"
LLM_TEMPERATURE = 0
LLM_NUM_PREDICT = 256


def get_local_ollama_models(base_url: str):
    with urlopen(f"{base_url}/api/tags", timeout=5) as resp:
        payload = json.loads(resp.read().decode("utf-8"))
    return [m["name"] for m in payload.get("models", [])]


installed_models = get_local_ollama_models(OLLAMA_BASE_URL)
print("Installed Ollama models:", installed_models)

# Prefer dedicated embedding models if available; otherwise fallback to a local LLM model.
embed_candidates = ["nomic-embed-text", "mxbai-embed-large", "bge-m3"]
llm_candidates = ["qwen2.5:7b", "llama3.2:latest", "llama3:latest", "gemma3:270m"]

EMBED_MODEL = next((m for m in embed_candidates if m in installed_models), None)
if EMBED_MODEL is None:
    EMBED_MODEL = next((m for m in llm_candidates if m in installed_models), installed_models[0])
    print("WARNING: No dedicated embedding model found.")
    print("For best retrieval, run: ollama pull nomic-embed-text")

LLM_MODEL = next((m for m in llm_candidates if m in installed_models), installed_models[0])

# Keep collection tied to embedding model to avoid mixing incompatible vectors.
COLLECTION_NAME = f"rag_collection_{EMBED_MODEL.replace(':', '_').replace('-', '_')}"

print("Embedding model:", EMBED_MODEL)
print("LLM model:", LLM_MODEL)
print("Collection:", COLLECTION_NAME)


Installed Ollama models: ['nomic-embed-text:latest', 'llama3:latest', 'qwen2.5:7b', 'gemma3:270m', 'llama3.2:latest']
For best retrieval, run: ollama pull nomic-embed-text
Embedding model: qwen2.5:7b
LLM model: qwen2.5:7b
Collection: rag_collection_qwen2.5_7b


In [50]:
sample_text = "Machine learning is fascinating"
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)
embeddings


OllamaEmbeddings(model='qwen2.5:7b', validate_model_on_init=False, base_url='http://localhost:11434', client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [51]:
import time
start = time.time()
vector = embeddings.embed_query(sample_text)
print(f"Embedding dimensions: {len(vector)}")
print(f"Embedding took: {time.time() - start:.2f}s")
vector[:10]


Embedding dimensions: 3584
Embedding took: 5.88s


[-0.0025533382,
 -0.0031807914,
 -0.007671313,
 -0.01324508,
 -0.007923292,
 -0.006662808,
 0.010056334,
 0.008781839,
 0.0007532049,
 0.0038406865]

### Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [52]:
chunks

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n   

In [53]:
## Create a ChromaDB vector store
persist_directory = "./chroma_db"

# Reset the specific collection to avoid stale vectors from previous runs.
client = chromadb.PersistentClient(path=persist_directory)
existing = {c.name for c in client.list_collections()}
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")

## Initialize ChromaDB with Ollama embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name=COLLECTION_NAME
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")


Deleted existing collection: rag_collection_qwen2.5_7b
Vector store created with 9 vectors
Persisted to: ./chroma_db


### Test Similarity Search

In [54]:
query="What are the types of machine learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs 

[Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.')]

In [55]:
query="what is NLP?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals')]

In [56]:
query="what is Deep Learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)')]

In [57]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: what is Deep Learning?

Top 3 similar chunks:

--- Chunk 1 ---
Deep Learning and Neural Networks...
Source: data/doc_1.txt

--- Chunk 2 ---
Machine Learning Fundamentals...
Source: data/doc_0.txt

--- Chunk 3 ---
Natural Language Processing (NLP)...
Source: data/doc_2.txt


### Advanced Similarity Search With Scores

In [58]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks'),
  0.5188323259353638),
 (Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals'),
  0.5581512451171875),
 (Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)'),
  0.595599889755249)]

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [59]:
llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=LLM_TEMPERATURE,
    num_predict=LLM_NUM_PREDICT,
)
llm


ChatOllama(model='qwen2.5:7b', num_predict=256, temperature=0.0, base_url='http://localhost:11434')

In [60]:
test_response = llm.invoke("What are large language models?")
test_response


AIMessage(content='Large Language Models (LLMs) are artificial intelligence systems designed to understand and generate human-like text based on the input they receive. These models are typically deep learning models that have been trained on vast amounts of textual data, allowing them to learn complex patterns in language.\n\nKey characteristics of large language models include:\n\n1. **Size**: LLMs can contain billions or even trillions of parameters, which allows them to capture intricate details and nuances in language.\n\n2. **Training Data**: They are often trained on diverse datasets that cover a wide range of topics and styles, enabling them to generate text across various domains.\n\n3. **Generative Capabilities**: LLMs can produce coherent and contextually relevant text, making them useful for tasks such as writing articles, generating code, creating stories, answering questions, and more.\n\n4. **Context Understanding**: These models are capable of understanding the context 

In [61]:
# Optional generic API demo (do NOT replace main llm used by RAG)
from langchain.chat_models.base import init_chat_model

llm_alt = init_chat_model(f"ollama:{LLM_MODEL}")
llm_alt


ChatOllama(model='qwen2.5:7b')

In [62]:
llm_alt.invoke("What is AI in one short sentence?")


AIMessage(content='AI is the development of computer systems to perform tasks that typically require human intelligence.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-02-13T01:27:10.213199Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4723791917, 'load_duration': 115906125, 'prompt_eval_count': 37, 'prompt_eval_duration': 534345375, 'eval_count': 17, 'eval_duration': 4035378252, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--019c549b-d80f-75d3-bf88-11e2006bab32-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 17, 'total_tokens': 54})

### Modern RAG Chain

In [63]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [64]:
## Convert vector store to retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 8, "lambda_mult": 0.5}
)
retriever


VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x169492450>, search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 8, 'lambda_mult': 0.5})

In [65]:
## Create a prompt template
from langchain_core.prompts import ChatPromptTemplate
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [66]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

##### What is create_stuff_documents_chain?
create_stuff_documents_chain creates a chain that "stuffs" (inserts) all retrieved documents into a single prompt and sends it to the LLM. It's called "stuff" because it literally stuffs all the documents into the context window at once.

In [67]:
### Create a document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOllama(model='qwen2.5:7b', num_predict=256, temperature=0.0, base_url='http://localhost:11434')
| StrOutputParser(), kwargs={}, 

This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

#### What is create_retrieval_chain?
create_retrieval_chain is a function that combines a retriever (which fetches relevant documents) with a document chain (which processes those documents with an LLM) to create a complete RAG pipeline.

In [68]:
### Create The Final RAG Chain
from langchain_classic.chains import create_retrieval_chain
rag_chain=create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x169492450>, search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 8, 'lambda_mult': 0.5}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of ret

In [69]:
response=rag_chain.invoke({"input":"What is Deep LEarning"})

In [70]:
response

{'input': 'What is Deep LEarning',
 'context': [Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals'),
  Document(metadata={'source': 'data/doc_2.txt'}, page_content='architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  Document(metadata={'source': 'data/doc_2.txt'}, page_content='NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer')],
 'answer': 'Deep Learning is a subset of machine learning that uses neural networks with multiple layers to learn hierarchical features from data. These multi-layered architectures enable deep learning models to automatically extract complex patterns and solve tasks like image recognition, natur

In [71]:
response['answer']

'Deep Learning is a subset of machine learning that uses neural networks with multiple layers to learn hierarchical features from data. These multi-layered architectures enable deep learning models to automatically extract complex patterns and solve tasks like image recognition, natural language processing, and more.'

### Compare LLMs: Llama 3.2 vs Qwen 2.5 7B
Run the same RAG question through both models to compare answer quality (same retriever and prompt).

In [72]:
# Same question for both models (same retriever + prompt, different LLM only)
from langchain_ollama import ChatOllama
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

compare_question = "What is Deep Learning?"
llm_llama = ChatOllama(model="llama3.2:latest", base_url=OLLAMA_BASE_URL, temperature=0, num_predict=256)
llm_qwen = ChatOllama(model="qwen2.5:7b", base_url=OLLAMA_BASE_URL, temperature=0, num_predict=256)

doc_chain_llama = create_stuff_documents_chain(llm_llama, prompt)
doc_chain_qwen = create_stuff_documents_chain(llm_qwen, prompt)
rag_llama = create_retrieval_chain(retriever, doc_chain_llama)
rag_qwen = create_retrieval_chain(retriever, doc_chain_qwen)

print("=== Llama 3.2 :latest ===")
r_llama = rag_llama.invoke({"input": compare_question})
print(r_llama["answer"])
print()
print("=== Qwen 2.5 7B ===")
r_qwen = rag_qwen.invoke({"input": compare_question})
print(r_qwen["answer"])
print()
print("Same retrieved context used for both (first doc preview):", r_llama["context"][0].page_content[:150], "...")

=== Llama 3.2 :latest ===
I don't know the answer to that question. However, I can tell you that Deep Learning is a subset of Machine Learning that involves the use of neural networks with multiple layers to analyze and interpret data. It's often used for tasks like image recognition, natural language processing, and speech recognition.

=== Qwen 2.5 7B ===
Deep Learning is a subset of machine learning that uses neural networks with multiple layers to learn complex representations from data, excelling in tasks like image and speech recognition.

Same retrieved context used for both (first doc preview): Deep Learning and Neural Networks ...


**How to check which model gives better results**

1. **By eye (best for you):** Read both answers above. Prefer the one that:
   - **Stays on topic** and answers the question.
   - **Uses the retrieved context** (matches your docs, doesn’t invent things).
   - **Is clear and concise** (no unnecessary fluff or repetition).

2. **Simple grounding check (below):** We run the same questions through both models and score how much of the retrieved context appears in each answer (rough “grounded in context” signal). Higher score = more tied to your docs.

In [73]:
# Run same questions through both models and compare (simple grounding check)
def simple_grounding_score(answer: str, context_docs: list) -> float:
    """Rough score 0-1: share of context words (min 3 chars) that appear in the answer."""
    words = set()
    for doc in context_docs:
        for w in doc.page_content.split():
            if len(w) >= 3:
                words.add(w.lower())
    if not words:
        return 0.0
    answer_lower = answer.lower()
    found = sum(1 for w in words if w in answer_lower)
    return round(found / len(words), 3)

test_questions = [
    "What is Deep Learning?",
    "What are the three types of machine learning?",
    "What is NLP?",
]

print("Model comparison (same retriever + prompt)\n" + "=" * 60)
for q in test_questions:
    r_llama = rag_llama.invoke({"input": q})
    r_qwen = rag_qwen.invoke({"input": q})
    a_llama, a_qwen = r_llama["answer"], r_qwen["answer"]
    ctx = r_llama["context"]
    g_llama = simple_grounding_score(a_llama, ctx)
    g_qwen = simple_grounding_score(a_qwen, ctx)
    print(f"\nQ: {q}")
    print(f"  Llama 3.2  | grounding: {g_llama} | {a_llama[:120]}...")
    print(f"  Qwen 2.5 7B| grounding: {g_qwen} | {a_qwen[:120]}...")
    better = "Llama" if g_llama > g_qwen else "Qwen" if g_qwen > g_llama else "tie"
    print(f"  -> higher grounding this run: {better}")
print("\n" + "=" * 60)
print("Conclusion: Prefer the model whose answers look better to you and stay closer to your docs. Grounding score is only a hint.")

Model comparison (same retriever + prompt)

Q: What is Deep Learning?
  Llama 3.2  | grounding: 0.48 | I don't know the answer to that question. However, I can tell you that Deep Learning is a subset of Machine Learning tha...
  Qwen 2.5 7B| grounding: 0.44 | Deep Learning is a subset of machine learning that uses neural networks with multiple layers to learn complex representa...
  -> higher grounding this run: Llama

Q: What are the three types of machine learning?
  Llama 3.2  | grounding: 0.517 | There are three main types of machine learning: Supervised, Unsupervised, and Reinforcement learning. These categories h...
  Qwen 2.5 7B| grounding: 0.793 | The three types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervis...
  -> higher grounding this run: Qwen

Q: What is NLP?
  Llama 3.2  | grounding: 0.417 | I don't know the answer to that question. However, I can tell you that Natural Language Processing (NLP) is a subfield o...


In [74]:
# Function to query the modern RAG system
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-" * 50)

    result = rag_chain.invoke({"input": question})

    print(f"Answer: {result['answer']}")
    print("\nRetrieved Context:")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

    return result

# Keep this short for local models (qwen2.5:7b on laptop can be slow)
test_questions = [
    "What are the three types of machine learning?"
]

for question in test_questions:
    result = query_rag_modern(question)
    print("\n" + "="*80 + "\n")


Question: What are the three types of machine learning?
--------------------------------------------------
Answer: The three types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through interaction with an environment using rewards and penalties.

Retrieved Context:

--- Source 1 ---
learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an envir...

--- Source 2 ---
Deep Learning and Neural Networks...

--- Source 3 ---
Machine Learning Fundamentals...




### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [75]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [76]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [77]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x169492450>, search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 8, 'lambda_mult': 0.5})

In [78]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [79]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x169492450>, search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 8, 'lambda_mult': 0.5})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOllama(model='qwen2.5:7b', num_predict=256, temperature=0.0, base_url='http://localhost:11434')
| StrOutputParser()

In [80]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

'Deep learning is a subset of machine learning that focuses on artificial neural networks, which are designed to mimic the functioning of the human brain. These networks consist of layers of interconnected nodes and have been instrumental in advancing various fields such as computer vision and natural language processing.\n\nSpecific details from the context supporting this answer include:\n- "Deep learning is a subset of machine learning based on artificial neural networks."\n- "These networks are inspired by the human brain and consist of layers of interconnected nodes."\n- "Deep learning has revolutionized fields like computer vision, natural language"'

In [81]:
retriever.invoke("What is Deep Learning")


[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers \n    excel at sequential data processing.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language')]

In [82]:
# Query using the LCEL approach
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Retrieve source docs (new API)
    docs = retriever.invoke(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

    return answer


In [83]:
# Test LCEL chain (single query to keep runtime short)
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")


Testing LCEL Chain:
Question: What are the key concepts in reinforcement learning?
--------------------------------------------------
Answer: Based on the context provided, the key concept in reinforcement learning is that it involves learning through interaction with an environment using rewards and penalties. Specifically:

- Reinforcement learning learns through interaction with an environment.
- It uses a system of rewards and penalties to guide the learning process.

This approach contrasts with supervised learning, which relies on labeled data, and unsupervised learning, which seeks patterns in unlabeled data.

Source Documents:

--- Source 1 ---
learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an envir...

--- Source 2 ---
Deep Learning and Neural Networks...

--- Source 3 ---
Machine learning is a subset of artificial intelligence that e

'Based on the context provided, the key concept in reinforcement learning is that it involves learning through interaction with an environment using rewards and penalties. Specifically:\n\n- Reinforcement learning learns through interaction with an environment.\n- It uses a system of rewards and penalties to guide the learning process.\n\nThis approach contrasts with supervised learning, which relies on labeled data, and unsupervised learning, which seeks patterns in unlabeled data.'

In [84]:
# Optional extra test
# query_rag_lcel("What is machine learning?")


In [85]:
# Optional extra test
# query_rag_lcel("What is deep learning?")


### Add New Documents To Existing Vector Store

In [86]:
vectorstore

In [87]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [88]:
new_document

'\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n'

In [89]:
chunks

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n   

In [90]:
new_doc=Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

In [91]:
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [92]:
## split the documents
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning in Detail'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [93]:
### Add new documents to vectorstore
vectorstore.add_documents(new_chunks)



['1589b71d-de88-40c9-8916-03e8f09de2cd',
 '00c8f5ba-1838-4885-81a8-7605b20b50d9',
 '8d634b1d-1bc3-4a84-b747-282fddadbc67']

In [94]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 3 new chunks to the vector store
Total vectors now: 12


In [95]:
## query with the updated vector
new_question="What are the keys concepts in reinforcement learning"
result=query_rag_lcel(new_question)
result

Question: What are the keys concepts in reinforcement learning
--------------------------------------------------
Answer: The key concepts in reinforcement learning (RL) include:

- States: These represent the current situation or condition of the environment as perceived by the agent.
- Actions: These are the possible choices that the agent can take to interact with its environment.
- Rewards: These are the feedback signals provided to the agent based on its actions, indicating how well it is performing towards achieving a goal.
- Policies: This refers to the strategy or rule that the agent uses to decide which action to take in different states.
- Value Functions: These quantify the expected utility of being in a particular state or taking an action in a given state.

These concepts are fundamental to understanding and implementing reinforcement learning algorithms.

Source Documents:

--- Source 1 ---
Reinforcement learning (RL) is a type of machine learning where an agent learns to

'The key concepts in reinforcement learning (RL) include:\n\n- States: These represent the current situation or condition of the environment as perceived by the agent.\n- Actions: These are the possible choices that the agent can take to interact with its environment.\n- Rewards: These are the feedback signals provided to the agent based on its actions, indicating how well it is performing towards achieving a goal.\n- Policies: This refers to the strategy or rule that the agent uses to decide which action to take in different states.\n- Value Functions: These quantify the expected utility of being in a particular state or taking an action in a given state.\n\nThese concepts are fundamental to understanding and implementing reinforcement learning algorithms.'

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [96]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [97]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [98]:
## create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x169492450>, search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 8, 'lambda_mult': 0.5}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[lan

In [99]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever, 
    question_answer_chain
)
print("Conversational RAG chain created!")

Conversational RAG chain created!


In [100]:
chat_history=[]
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a subset of artificial intelligence that involves training algorithms to make predictions or decisions without being explicitly programmed. It uses statistical techniques to find patterns in data, allowing systems to improve their performance on a task with experience.


In [101]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [102]:
chat_history

[HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a subset of artificial intelligence that involves training algorithms to make predictions or decisions without being explicitly programmed. It uses statistical techniques to find patterns in data, allowing systems to improve their performance on a task with experience.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [103]:
## Follow up question
# Follow-up question
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
result2

{'chat_history': [HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a subset of artificial intelligence that involves training algorithms to make predictions or decisions without being explicitly programmed. It uses statistical techniques to find patterns in data, allowing systems to improve their performance on a task with experience.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'input': 'What are its main types?',
 'context': [Document(metadata={'source': 'data/doc_0.txt'}, page_content='learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
  Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks'),
  Document(metadata={'topic': 'reinf

In [104]:
result2['answer']

'The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data for training, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through interaction with an environment using rewards and penalties.'

### Using GROQ LLM's
 

In [105]:
llm

ChatOllama(model='qwen2.5:7b', num_predict=256, temperature=0.0, base_url='http://localhost:11434')

In [106]:
# Optional: switch model quickly (must exist in `ollama list`)
# Example alternatives from your machine: "llama3:latest", "llama3.2:latest", "qwen2.5:7b"
# LLM_MODEL = "llama3.2:latest"
print("Current LLM_MODEL:", LLM_MODEL)


Current LLM_MODEL: qwen2.5:7b


In [107]:
# Quick health check for local Ollama service
from urllib.request import urlopen

try:
    with urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=5) as resp:
        print("Ollama reachable:", resp.status == 200)
except Exception as e:
    print("Ollama not reachable:", e)


Ollama reachable: True


In [108]:
from langchain_ollama import ChatOllama
from langchain.chat_models import init_chat_model


In [109]:
llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=LLM_TEMPERATURE,
    num_predict=LLM_NUM_PREDICT,
)
llm


ChatOllama(model='qwen2.5:7b', num_predict=256, temperature=0.0, base_url='http://localhost:11434')

In [110]:
llm.invoke("Explain reinforcement learning in 2 short lines")


AIMessage(content='Reinforcement learning is a type of machine learning where an agent learns to behave in an environment by performing actions and receiving rewards or penalties. The goal is to maximize the cumulative reward over time.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-02-13T01:31:45.896574Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8205254417, 'load_duration': 117399583, 'prompt_eval_count': 38, 'prompt_eval_duration': 847023166, 'eval_count': 40, 'eval_duration': 7209965796, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--019c549f-ff59-7773-b815-01af73229a8d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 40, 'total_tokens': 78})

In [111]:
# Optional generic API demo again
llm_alt = init_chat_model(model=f"ollama:{LLM_MODEL}")
llm_alt


ChatOllama(model='qwen2.5:7b')